# 07 — Data gaps audit: does the open-data slice reproduce the `hinnang` label?

**Context.** `docs/report.md` §7 and `docs/ml_framing.md` §1 both state that the `compliant` label is a deterministic function of the measured parameters against the norms printed in `docs/normy.md`. If that premise holds, **every** probe in `load_all()` should be classifiable by re-applying those norms directly — no ML needed.

**Question.** For how many probes does the deterministic norm check actually **disagree** with the official `compliant` label? Where are the disagreements concentrated (domain, year, parameter coverage)?

**Buckets** (see `src/audit/label_vs_norms.py`):

| Bucket | label | checker | Interpretation |
|---|---|---|---|
| `agree_pass` | compliant | no violation | Routine — label reproducible. |
| `agree_violate` | violation | violation | Routine — label reproducible. |
| `hidden_violation` | violation | no violation | **The core signal.** Label says non-compliant but no published parameter exceeds its norm. Direct evidence for the reflection-note hypothesis that the open-data slice is a subset of the internal record. |
| `hidden_pass` | compliant | violation | Threshold interpretation / aggregation rule gap (e.g. EU 2006/7/EC bathing-water 95th-percentile rule for `supluskoha`). |
| `unknown` | NaN | — | `hinnang` absent in source XML. |

**Reference:** `reflect/2026-04-15_health-data-gaps.md` in `sapsan14/life`. See also `docs/data_gaps.md` for the synthesis.

**Caveat.** The bathing-water directive 2006/7/EC classifies sites on a multi-probe 90/95-percentile; our check is per-probe. `hidden_pass` on `supluskoha` is therefore partly expected and must be reported separately.

In [ ]:
import sys
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

# When the project is installed editable (`pip install -e .`), src/ is on sys.path.
from data_loader import load_all
from audit.label_vs_norms import audit_dataframe, BUCKETS

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1. Load all four opendata domains

In [ ]:
df = load_all()
print(f"Total probes: {len(df):,}")
print(f"With known compliant: {df['compliant'].notna().sum():,}")
print("\nDomain counts:")
print(df['domain'].value_counts())

## 2. Run the deterministic checker

In [ ]:
audited = audit_dataframe(df)
print("Bucket distribution (all domains, all probes):")
print(audited['bucket'].value_counts())
print("\nAs share of probes with known label:")
known = audited[audited['compliant'].notna()]
print((known['bucket'].value_counts(normalize=True) * 100).round(2).astype(str) + ' %')

### Sanity check against the report

Per `docs/report.md` §3.1, 92.9% of probes are compliant and ~7.1% are violations. If the deterministic checker is faithful to `features.add_ratio_features`, the `agree_*` buckets together should dominate — ideally > 90%. Anything lower indicates the checker has drifted from the feature logic and the audit must not be trusted until fixed.

In [ ]:
agree_share = ((audited['bucket'].isin(['agree_pass', 'agree_violate'])).sum() / max(1, len(known))) * 100
print(f"agree_pass + agree_violate covers {agree_share:.1f}% of labelled probes")
assert agree_share >= 85.0, (
    f"Checker agreement too low ({agree_share:.1f}%). "
    f"Likely drift from features.add_ratio_features — investigate before publishing."
)

## 3. Breakdown by domain × bucket

`supluskoha` is reported separately because the bathing-water 95th-percentile rule means per-probe `hidden_pass` is not necessarily a data-gap finding.

In [ ]:
by_domain = pd.crosstab(audited['domain'], audited['bucket'])
# Guarantee all bucket columns exist even if empty.
for b in BUCKETS:
    if b not in by_domain.columns:
        by_domain[b] = 0
by_domain = by_domain[list(BUCKETS)]
by_domain['total'] = by_domain.sum(axis=1)
by_domain

In [ ]:
# Rates of the two gap buckets per domain, over known-label probes only.
labelled = audited[audited['compliant'].notna()]
rates = (
    labelled.groupby('domain')['bucket']
    .value_counts(normalize=True)
    .unstack(fill_value=0.0)
    .mul(100)
    .round(2)
)
for b in BUCKETS:
    if b not in rates.columns:
        rates[b] = 0.0
rates[list(BUCKETS)]

## 4. Hidden violations — the core signal

For every probe in the `hidden_violation` bucket, list the parameters that have a norm but were **not measured**. If the same parameters are missing across the whole bucket, that's a strong hint about which fields are withheld or selectively measured.

In [ ]:
hv = audited[audited['bucket'] == 'hidden_violation'].copy()
print(f"Hidden violations: {len(hv):,}")
if len(hv):
    print("\nBy domain:")
    print(hv['domain'].value_counts())

    print("\nBy year:")
    hv['year'] = pd.to_datetime(hv['sample_date']).dt.year
    print(hv['year'].value_counts().sort_index())

    print("\nMean number of measured norm-params per hidden_violation probe:")
    print(f"  {hv['n_measured_norm_params'].mean():.2f} (full dataset mean: {audited['n_measured_norm_params'].mean():.2f})")

In [ ]:
# Which norm-parameters are most commonly unmeasured inside hidden_violation?
if len(hv):
    from collections import Counter
    missing_counter = Counter()
    for unmeasured in hv['unmeasured_norm_params']:
        missing_counter.update(unmeasured)
    unmeasured_series = pd.Series(missing_counter).sort_values(ascending=False)
    unmeasured_series.to_frame('count').assign(
        pct_of_hidden_violations=lambda d: (d['count'] / len(hv) * 100).round(1)
    )
else:
    print('No hidden_violation probes — nothing to profile.')

## 5. Per-location tally using `location_key`

`location_key` is the normalised site identifier from `data_loader.normalize_location()` — use it rather than raw `location` to avoid double-counting renamed sites (see CLAUDE.md).

In [ ]:
if len(hv):
    top_sites = (
        hv.groupby(['domain', 'location_key'])
        .size()
        .rename('hidden_violation_count')
        .sort_values(ascending=False)
        .head(25)
    )
    print(top_sites)

## 6. Hidden passes (aggregate / threshold rule gaps)

Report `supluskoha` separately — the 95th-percentile bathing-water rule can legitimately explain a per-probe spike staying officially compliant.

In [ ]:
hp = audited[audited['bucket'] == 'hidden_pass']
print(f"Hidden passes: {len(hp):,}")
if len(hp):
    print('\nBy domain:')
    print(hp['domain'].value_counts())

    print('\nMost common violated params (hidden_pass):')
    from collections import Counter
    c = Counter()
    for vs in hp['violated_params']:
        c.update(vs)
    print(pd.Series(c).sort_values(ascending=False).head(15))

## 7. Export the audit artifact

`data/audit/divergences_{YYYY-MM-DD}.parquet` is the reproducibility anchor for `docs/data_gaps.md` and for the draft engineering inquiry in `docs/terviseamet_inquiry.md`. Keeps probe-level rows for both gap buckets.

In [ ]:
out_dir = Path('..') / 'data' / 'audit'
out_dir.mkdir(parents=True, exist_ok=True)
divergences = audited[audited['bucket'].isin(['hidden_violation', 'hidden_pass'])].copy()

# Parquet doesn't serialise list columns well — stringify them for the artifact.
divergences['violated_params'] = divergences['violated_params'].apply(lambda xs: ','.join(xs))
divergences['unmeasured_norm_params'] = divergences['unmeasured_norm_params'].apply(lambda xs: ','.join(xs))

out_path = out_dir / f"divergences_{date.today().isoformat()}.parquet"
try:
    divergences.to_parquet(out_path, index=False)
    print(f"Wrote {len(divergences):,} rows to {out_path}")
except Exception as exc:
    # Fall back to CSV if pyarrow/fastparquet is unavailable.
    csv_path = out_path.with_suffix('.csv')
    divergences.to_csv(csv_path, index=False)
    print(f"parquet unavailable ({exc}); wrote {len(divergences):,} rows to {csv_path}")

## 8. Intersection with the model's confusion set (optional)

Per `docs/report.md` §5.2 the best model has 30 FN and 25 FP on the random-split test set. Most of those errors should land in the `hidden_*` buckets: if a probe is mislabelled in the data, the model has no way to get it right. Load `models/best_model.joblib` and check the overlap — this re-attributes the model's residual errors to data gaps rather than model limitations.

In [ ]:
import joblib

best_path = Path('..') / 'models' / 'best_model.joblib'
if best_path.exists():
    bundle = joblib.load(best_path)
    print(f'Loaded bundle keys: {list(bundle.keys()) if isinstance(bundle, dict) else type(bundle)}')
    # Left as a reader exercise: align bundle['X_test'] indices with audited rows
    # by sample_id and compute the share of FN/FP sitting in hidden_* buckets.
else:
    print('models/best_model.joblib not found — run notebook 06 first to enable this analysis.')